# Strategic RL staged-arm report

Control, behavior-KL, categorical critic, and state-curriculum runs are compared only on identical evaluation keys: opponent, seed, map size, and orientation. Evaluation remains turn-0 only.

In [7]:
from pathlib import Path
import json
import pandas as pd

RUNS = {  # point each arm at a fresh evaluation directory
    'control': Path('outputs/eval_control'),
    'behavior_kl': Path('outputs/eval_behavior_kl'),
    'categorical': Path('outputs/eval_categorical'),
    'curriculum': Path('outputs/eval_curriculum'),
}

frames = []
for arm, run_dir in RUNS.items():
    games = run_dir / 'games.jsonl'
    if games.is_file():
        frame = pd.read_json(games, lines=True)
        frame['arm'] = arm
        frames.append(frame)
games = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
games


,backend,candidate_city_survival,candidate_final_city_tiles,candidate_final_units,candidate_player,candidate_stranded_fuel,map_size,match_id,opponent,seed,winner,arm,candidate_digest,rng_id,rng_scheme
0,internal_batched,0.854988,24,24,0,0.079404,12,seed-2021-size-12-p0,first_place,2021,0,control,NaN,NaN,NaN
1,internal_batched,0.856230,21,3,1,0.081005,12,seed-2021-size-12-p1,first_place,2021,1,control,NaN,NaN,NaN
2,internal_batched,0.818729,24,24,0,0.081412,16,seed-2021-size-16-p0,first_place,2021,1,control,NaN,NaN,NaN
3,internal_batched,0.620513,8,9,1,0.152457,16,seed-2021-size-16-p1,first_place,2021,0,control,NaN,NaN,NaN
4,internal_batched,0.670871,19,19,0,0.120422,24,seed-2021-size-24-p0,first_place,2021,1,control,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,internal_batched,0.739728,54,16,1,0.196056,16,seed-2030-size-16-p1,first_place,2030,1,behavior_kl,c9aded84904165a60ec1485ccd4760f16c1b094f2b49f4...,3319bed9119743fc,lux-internal-v2:seed-map-orientation
156,internal_batched,0.632605,25,30,0,0.142779,24,seed-2030-size-24-p0,first_place,2030,1,behavior_kl,c9aded84904165a60ec1485ccd4760f16c1b094f2b49f4...,1f71a1e51108165e,lux-internal-v2:seed-map-orientation
157,internal_batched,0.735170,182,182,1,0.072387,24,seed-2030-size-24-p1,first_place,2030,1,behavior_kl,c9aded84904165a60ec1485ccd4760f16c1b094f2b49f4...,a983cc4e722ed012,lux-internal-v2:seed-map-orientation
158,internal_batched,0.675366,237,237,0,0.076669,32,seed-2030-size-32-p0,first_place,2030,1,behavior_kl,c9aded84904165a60ec1485ccd4760f16c1b094f2b49f4...,802bd9ecd7d63dd3,lux-internal-v2:seed-map-orientation


In [8]:
KEYS = ['opponent', 'seed', 'map_size', 'candidate_player']
if not games.empty:
    games['score'] = (games.winner == games.candidate_player).astype(float) + 0.5 * (games.winner == -1)
    duplicate_keys = games.duplicated(['arm', *KEYS], keep=False)
    assert not duplicate_keys.any(), games.loc[duplicate_keys, ['arm', *KEYS]]
    display(games.groupby(['arm', 'opponent', 'map_size', 'candidate_player']).agg(
        score=('score', 'mean'), games=('score', 'size'),
        city_survival=('candidate_city_survival', 'mean'),
        final_city=('candidate_final_city_tiles', 'mean'),
        final_units=('candidate_final_units', 'mean'),
    ))

score  games  \
arm         opponent    map_size candidate_player                 
behavior_kl first_place 12       0                   0.4     10   
                                 1                   0.4     10   
                        16       0                   0.3     10   
                                 1                   0.2     10   
                        24       0                   0.1     10   
                                 1                   0.7     10   
                        32       0                   0.3     10   
                                 1                   0.1     10   
control     first_place 12       0                   0.4     10   
                                 1                   0.4     10   
                        16       0                   0.2     10   
                                 1                   0.1     10   
                        24       0                   0.1     10   
                                 1                   0.2     10   
                        32       0                   0.2     10   
                                 1                   0.1     10   

                                                   city_survival  final_city  \
arm         opponent    map_size candidate_player                              
behavior_kl first_place 12       0                      0.724477        11.4   
                                 1                      0.751679        22.4   
                        16       0                      0.794117        36.3   
                                 1                      0.723151        27.4   
                        24       0                      0.717978        62.0   
                                 1                      0.808331       119.6   
                        32       0                      0.645163        85.4   
                                 1                      0.712948       108.4   
control     first_place 12       0                      0.702984        13.1   
                                 1                      0.689775        16.5   
                        16       0                      0.832693        42.4   
                                 1                      0.699592        25.1   
                        24       0                      0.717600        46.3   
                                 1                      0.759759        97.8   
                        32       0                      0.702106       139.5   
                                 1                      0.722410       136.1   

                                                   final_units  
arm         opponent    map_size candidate_player               
behavior_kl first_place 12       0                        12.9  
                                 1                        22.7  
                        16       0                        36.5  
                                 1                        23.9  
                        24       0                        64.6  
                                 1                       119.6  
                        32       0                        89.0  
                                 1                       110.1  
control     first_place 12       0                        14.5  
                                 1                        15.7  
                        16       0                        42.7  
                                 1                        23.2  
                        24       0                        47.5  
                                 1                        97.9  
                        32       0                       141.4  
                                 1                       137.0

## Learner diagnostics

Export flattened learner history to `metrics.jsonl`. Required keys include `Behavior_Policy.*`, `Normalized_Entropy.*`, `Active_Entities.*`, `Value.*`, and `Gradient.*`. A categorical finalist is invalid while `Value.support_outside_fraction > 0.001`. Promotion also requires the paired bootstrap and survival gates defined in the run report.

In [9]:
metric_frames = []
for arm, run_dir in RUNS.items():
    path = run_dir / 'metrics.jsonl'
    if path.is_file():
        frame = pd.read_json(path, lines=True)
        frame['arm'] = arm
        metric_frames.append(frame)
metrics = pd.concat(metric_frames, ignore_index=True) if metric_frames else pd.DataFrame()
metrics

""
